## 介绍


在本笔记本中，我们将"从零开始"构建并训练一个深度学习模型——也就是说，我们不会使用任何预先构建的网络结构、优化器或数据加载框架等。

我们假设你已经了解神经网络的基本工作原理。如果还不了解，请先阅读这篇笔记本：[神经网络到底是如何工作的？](https://www.kaggle.com/code/jhoward/how-does-a-neural-net-really-work)。本笔记本将使用 Kaggle 的 [Titanic](https://www.kaggle.com/competitions/titanic/) 竞赛数据集，因为它非常小巧简单，同时也涵盖了大多数实际项目中需要处理的许多棘手问题。（不过请注意，这个竞赛是 Kaggle 上的小型"入门级"竞赛，所以暂时不要期望神经网络能带来多大提升；等我们尝试真正的竞赛时，效果自然会体现出来！）

能够在自己的机器、Colab 以及 Kaggle 上运行同一个笔记本是非常方便的。为此，我们使用以下代码，在非 Kaggle 环境下按需下载数据（有关此技术的详细说明，请参阅[这篇笔记本](https://www.kaggle.com/code/jhoward/getting-started-with-nlp-for-absolute-beginners/)）：


In [ ]:
import os
from pathlib import Path

iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
if iskaggle: path = Path('../input/titanic')
else:
    path = Path('titanic')
    if not path.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(path))
        zipfile.ZipFile(f'{path}.zip').extractall(path)

请注意，Kaggle 比赛的数据始终存放在 `../input` 文件夹中。获取路径最简便的方法是点击 Kaggle 笔记本右上角的"K"按钮，点击显示的文件夹，然后点击复制按钮。

本笔记本将使用 *numpy* 和 *pytorch* 进行数组计算，使用 *pandas* 处理表格数据，因此我们需要导入它们，并将显示宽度设置得比默认值稍大一些。


In [ ]:
import torch, numpy as np, pandas as pd
np.set_printoptions(linewidth=140)
torch.set_printoptions(linewidth=140, sci_mode=False, edgeitems=7)
pd.set_option('display.width', 140)

## 清理数据


这是一个<em>表格数据</em>竞赛——数据以表格形式呈现。它以逗号分隔值（CSV）文件的形式提供。我们可以使用 *pandas* 库打开它，该库将创建一个 `DataFrame`。


In [ ]:
df = pd.read_csv(path/'train.csv')
df

正如我们在 <em>神经网络的真正工作原理</em> 笔记本中所学到的，我们需要将每一列乘以某些系数。但我们可以看到，`Cabin` 列中存在 `NaN` 值——这是 Pandas 表示缺失值的方式。我们无法对缺失值进行乘法运算！

让我们来检查哪些列包含 `NaN` 值。Pandas 的 `isna()` 函数会对 `NaN` 值返回 `True`（在作为数字使用时被视为 `1`），因此我们只需对每一列的结果求和即可：


In [ ]:
df.isna().sum()

请注意，Pandas 默认是对列进行求和。

我们需要用某个值来填充缺失值。通常选什么值影响不大，这里我们使用出现频率最高的值（即"<em>众数</em>"）。可以用 `mode` 函数来实现。需要注意的是，当存在多个并列众数时，它会返回多行，因此我们用 `iloc[0]` 取第一行：


In [ ]:
modes = df.mode().iloc[0]
modes

顺便说一句，在不理解函数的情况下使用它们从来都不是个好主意。所以请务必搜索你不熟悉的内容。例如，如果你想了解 `iloc`（这确实是一个非常重要的函数！），Google 会给你一个[很棒的教程](https://www.shanelynn.ie/pandas-iloc-loc-select-rows-and-columns-dataframe/)的链接。

现在我们已经得到了每一列的众数，可以使用 `fillna` 将缺失值替换为各列的众数。我们将"就地"完成这个操作——也就是说，我们会直接修改数据框本身，而不是返回一个新的数据框。


In [ ]:
df.fillna(modes, inplace=True)

我们现在可以检查是否还有缺失值：


In [ ]:
df.isna().sum()

以下是我们如何快速获取数据集中所有数值列的摘要：


In [ ]:
import numpy as np

df.describe(include=(np.number))

我们可以看到，`Fare` 字段的值主要集中在 `0` 到 `30` 之间，但也有一些非常大的值。这在包含金额的字段中非常常见，可能会给模型带来问题——因为当该列乘以某个系数时，那几个极大值所在的行将主导最终结果。

从直方图中可以最直观地看出这个问题：图中右侧有一条很长的尾巴（另外提醒一下：如果你对直方图还不太熟悉，可以谷歌"[histogram tutorial](https://www.google.com/search?q=histogram+tutorial&oq=histogram+tutorial)"，先阅读一些相关资料再继续）：


In [ ]:
df['Fare'].hist();

为了解决这个问题，最常见的方法是取对数，这样可以压缩较大的数值，使分布更加合理。但请注意，`Fare` 列中存在零值，而 `log(0)` 是无穷大——为了解决这个问题，我们只需先将所有值加 `1`：


In [ ]:
df['LogFare'] = np.log(df['Fare']+1)

直方图现在显示出更均匀的值分布，没有长尾：


In [ ]:
df['LogFare'].hist();

从`describe()`的输出来看，`Pclass`只包含3个值，我们可以通过查看[数据字典](https://www.kaggle.com/competitions/titanic/data)来确认这一点（对于任何项目，你都应该仔细研究数据字典！）——


In [ ]:
pclasses = sorted(df.Pclass.unique())
pclasses

以下是我们如何快速汇总数据集中所有非数值列的方法：


In [ ]:
df.describe(include=[object])

显然，我们无法将 `male` 或 `S` 这样的字符串乘以系数，因此需要将它们替换为数字。

我们通过创建包含<em>哑变量</em>的新列来实现这一点。哑变量是这样一种列：当某一列包含特定值时，该列的值为 `1`，否则为 `0`。例如，我们可以为 `Sex='male'` 创建一个哑变量——这是一个新列，在 `Sex` 为 `'male'` 的行中值为 `1`，在其他行中值为 `0`。

Pandas 可以使用 `get_dummies` 自动创建这些哑变量，同时也会移除原始列。我们将为 `Pclass` 创建哑变量，尽管它是数值类型，但数字 `1`、`2`、`3` 分别对应一等、二等和三等舱——而非可以直接相乘的计数或度量值。我们还将为 `Sex` 和 `Embarked` 创建哑变量，因为我们希望将它们作为模型中的预测变量。另一方面，`Cabin`、`Name` 和 `Ticket` 的唯一值太多，为它们创建哑变量并不合适。


In [ ]:
df = pd.get_dummies(df, columns=["Sex","Pclass","Embarked"])
df.columns

我们可以看到，末尾新增了 5 列——对应我们所请求的三列中每个可能值各占一列，同时那三列原始列已被移除。

以下是这些新增列前几行的内容：


In [ ]:
added_cols = ['Sex_male', 'Sex_female', 'Pclass_1', 'Pclass_2', 'Pclass_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
df[added_cols].head()

现在我们可以创建我们的自变量（预测变量）和因变量（目标变量）。它们都需要是 PyTorch 张量。我们的因变量是 `Survived`：


In [ ]:
from torch import tensor

t_dep = tensor(df.Survived)

我们的自变量是所有感兴趣的连续变量加上我们刚刚创建的所有虚拟变量：


In [ ]:
indep_cols = ['Age', 'SibSp', 'Parch', 'LogFare'] + added_cols

t_indep = tensor(df[indep_cols].values, dtype=torch.float)
t_indep

以下是我们自变量的行数和列数：


In [ ]:
t_indep.shape

## 建立线性模型


现在我们已经有了一个自变量矩阵和一个因变量向量，可以开始计算预测值和损失了。在本节中，我们将手动对数据的每一行执行一步预测值和损失的计算。

我们的第一个模型将是一个简单的线性模型。我们需要为 `t_indep` 中的每一列分配一个系数。我们将在 `(-0.5,0.5)` 范围内随机选取数值，并设置手动随机种子，以确保本笔记本中的说明与你运行时看到的结果保持一致。


In [ ]:
torch.manual_seed(442)

n_coeff = t_indep.shape[1]
coeffs = torch.rand(n_coeff)-0.5
coeffs

我们的预测值将通过将每一行与系数相乘后求和来计算。这里有一个有趣的地方：我们不需要单独的常数项（也称为"偏置"或"截距"项），也不需要一列全为 `1` 的数据来达到同样的效果。这是因为我们的虚拟变量已经覆盖了整个数据集——例如，有一列表示"男性"，另一列表示"女性"，数据集中的每个人恰好属于其中一列；因此，我们不需要单独的截距项来处理那些不属于任何列的行。

下面是乘法运算的具体过程：


In [ ]:
t_indep*coeffs

我们可以看到这里存在一个问题。每一行的总和将由第一列主导，也就是 `Age` 列，因为它的平均值比其他所有列都要大。

让我们通过将每一列除以其 `max()`，使所有列的数值都在 `0` 到 `1` 之间：


In [ ]:
vals,indices = t_indep.max(dim=0)
t_indep = t_indep / vals

正如我们所见，这消除了一列主导所有其他列的问题：


In [ ]:
t_indep*coeffs

你可能已经注意到，下面这行代码简直酷炫至极：

    t_indep = t_indep / vals

这是用一个矩阵除以一个向量——这到底是什么意思？！其实，这里我们利用了 numpy 和 PyTorch（以及许多其他语言，甚至可以追溯到 APL）中一种叫做[广播](https://numpy.org/doc/stable/user/basics.broadcasting.html)的技术。简单来说，它的效果就像是为矩阵的每一行都单独复制了一份向量，然后用每一行去除以这个向量。实际上，它并不会真的进行任何复制，而是以一种高度优化的方式完成整个运算，充分发挥现代 CPU（当然，如果使用 GPU 的话也同样适用）的性能。广播是让代码简洁、易维护且高效运行的最重要技巧之一，非常值得深入学习和练习。

现在，我们可以通过对乘积的每一行求和，从线性模型中生成预测值：


In [ ]:
preds = (t_indep*coeffs).sum(axis=1)

让我们来看看前几个：


In [ ]:
preds[:10]

当然，这些预测结果不会有什么实际用处，因为我们的系数是随机的——它们只是梯度下降过程的一个起点。

要进行梯度下降，我们需要一个损失函数。通常，取各行误差的平均值（即预测值与目标值之差的绝对值）是一种比较合理的方法：


In [ ]:
loss = torch.abs(preds-t_dep).mean()
loss

现在我们已经测试了一种计算预测和损失的方法，让我们把它们放入函数中以使工作更简单：


In [ ]:
def calc_preds(coeffs, indeps): return (indeps*coeffs).sum(axis=1)
def calc_loss(coeffs, indeps, deps): return torch.abs(calc_preds(coeffs, indeps)-deps).mean()

## 执行一步梯度下降


在本节中，我们将手动执行单次梯度下降的"epoch"。我们唯一要自动化的是计算梯度，因为说实话，手动计算梯度既繁琐又毫无意义！为了让 PyTorch 计算梯度，我们需要在 `coeffs` 上调用 `requires_grad_()`（如果你不确定原因，请在继续之前回顾上一个笔记本，[神经网络究竟是如何工作的？](https://www.kaggle.com/code/jhoward/how-does-a-neural-net-really-work)）：


In [ ]:
coeffs.requires_grad_()

现在当我们计算损失时，PyTorch 将跟踪所有步骤，因此我们之后能够获取梯度：


In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss

使用 `backward()` 让 PyTorch 现在计算梯度：


In [ ]:
loss.backward()

让我们看看它们是什么样子的：


In [ ]:
coeffs.grad

请注意，每次我们调用 `backward` 时，梯度实际上是<em>累加</em>到 `.grad` 属性中的。让我们再次尝试运行上述步骤：


In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
coeffs.grad

如你所见，我们的 `.grad` 值翻倍了。这是因为梯度被累加了第二次。正因如此，在我们使用梯度执行一步梯度下降之后，需要将其重置为零。

现在我们可以执行一步梯度下降，并验证损失是否有所下降：


In [ ]:
loss = calc_loss(coeffs, t_indep, t_dep)
loss.backward()
with torch.no_grad():
    coeffs.sub_(coeffs.grad * 0.1)
    coeffs.grad.zero_()
    print(calc_loss(coeffs, t_indep, t_dep))

请注意，`a.sub_(b)` 会从 `a` 中原地减去 `b`。在 PyTorch 中，任何以 `_` 结尾的方法都会原地修改其对象。类似地，`a.zero_()` 会将张量的所有元素设置为零。


## 训练线性模型


在开始训练模型之前，我们需要确保留出一个验证集来计算评估指标（详情请参阅"[NLP 零基础入门](https://www.kaggle.com/code/jhoward/getting-started-with-nlp-for-absolute-beginners#Test-and-validation-sets)"）。

实现这一目标的方法有很多。在下一个笔记本中，我们将把这里的做法与 fastai 库的方式进行对比，因此我们需要确保以相同的方式划分数据。那么，让我们使用 `RandomSplitter` 来获取索引，将数据划分为训练集和验证集：


In [ ]:
from fastai.data.transforms import RandomSplitter
trn_split,val_split=RandomSplitter(seed=42)(df)

现在我们可以将这些索引应用于我们的自变量和因变量：


In [ ]:
trn_indep,val_indep = t_indep[trn_split],t_indep[val_split]
trn_dep,val_dep = t_dep[trn_split],t_dep[val_split]
len(trn_indep),len(val_indep)

我们将为上面手动完成的三件事创建函数：更新 `coeffs`、执行一次完整的梯度下降步骤，以及将 `coeffs` 初始化为随机数：


In [ ]:
def update_coeffs(coeffs, lr):
    coeffs.sub_(coeffs.grad * lr)
    coeffs.grad.zero_()

In [ ]:
def one_epoch(coeffs, lr):
    loss = calc_loss(coeffs, trn_indep, trn_dep)
    loss.backward()
    with torch.no_grad(): update_coeffs(coeffs, lr)
    print(f"{loss:.3f}", end="; ")

In [ ]:
def init_coeffs(): return (torch.rand(n_coeff)-0.5).requires_grad_()

我们现在可以使用这些函数来训练我们的模型：


In [ ]:
def train_model(epochs=30, lr=0.01):
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): one_epoch(coeffs, lr=lr)
    return coeffs

让我们试试吧。我们的损失值将在每一步结束时打印出来，所以我们希望能看到它不断下降：


In [ ]:
coeffs = train_model(18, lr=0.2)

确实如此！

我们来看看每一列对应的系数：


In [ ]:
def show_coeffs(): return dict(zip(indep_cols, coeffs.requires_grad_(False)))
show_coeffs()

## 测量精度


Kaggle 竞赛的评分标准并不是绝对误差（即我们的损失函数），而是以<em>准确率</em>来评分——即我们正确预测生存结果的行数比例。让我们来看看在验证集上的准确率。首先，计算预测值：


In [ ]:
preds = calc_preds(coeffs, val_indep)

我们假设任何得分超过 `0.5` 的乘客都被预测为生还。因此，这意味着对于每一行，当 `preds>0.5` 与因变量相同时，我们的预测是正确的：


In [ ]:
results = val_dep.bool()==(preds>0.5)
results[:16]

让我们看看我们的平均准确率是：


In [ ]:
results.float().mean()

这不是一个坏的开始！我们将创建一个函数，以便我们可以轻松计算我们训练的其他模型的准确性：


In [ ]:
def acc(coeffs): return (val_dep.bool()==(calc_preds(coeffs, val_indep)>0.5)).float().mean()
acc(coeffs)

## 使用 sigmoid


看看我们的预测，有一个明显的问题——我们对生存概率的一些预测值 `>1`，而另一些则 `<0`：


In [ ]:
preds[:28]

为了解决这个问题，我们应该将每个预测值通过<em>sigmoid 函数</em>，该函数的最小值为零，最大值为一，定义如下：


In [ ]:
import sympy
sympy.plot("1/(1+exp(-x))", xlim=(-5,5));

PyTorch 已经为我们定义了该函数，所以我们可以修改 `calc_preds` 来使用它：


In [ ]:
def calc_preds(coeffs, indeps): return torch.sigmoid((indeps*coeffs).sum(axis=1))

现在让我们训练一个新模型，使用这个更新后的函数来计算预测：


In [ ]:
coeffs = train_model(lr=100)

损失已经大幅改善。让我们检查一下准确率：


In [ ]:
acc(coeffs)

这也改进了！以下是我们训练模型的系数：


In [ ]:
show_coeffs()

这些系数看起来是合理的——总体而言，年龄较大的人和男性生存的可能性较低，而头等舱乘客生存的可能性较高。


## 提交到 Kaggle


现在我们已经有了一个训练好的模型，我们可以准备向 Kaggle 提交结果了。为此，首先我们需要读取测试集：


In [ ]:
tst_df = pd.read_csv(path/'test.csv')

在这种情况下，测试集中有一名乘客缺少 `Fare` 数据。我们只需将其填充为 `0` 以避免问题：


In [ ]:
tst_df['Fare'] = tst_df.Fare.fillna(0)

现在我们可以复制我们在训练集上执行的相同步骤，并对测试集执行完全相同的操作来预处理数据：


In [ ]:
tst_df.fillna(modes, inplace=True)
tst_df['LogFare'] = np.log(tst_df['Fare']+1)
tst_df = pd.get_dummies(tst_df, columns=["Sex","Pclass","Embarked"])

tst_indep = tensor(tst_df[indep_cols].values, dtype=torch.float)
tst_indep = tst_indep / vals

让我们计算一下测试集中哪些乘客幸存的预测结果：


In [ ]:
tst_df['Survived'] = (calc_preds(tst_indep, coeffs)>0.5).int()

Kaggle竞赛网站上的示例提交文件显示，我们需要上传一个只包含 `PassengerId` 和 `Survived` 的CSV文件，所以让我们创建并保存它：


In [ ]:
sub_df = tst_df[['PassengerId','Survived']]
sub_df.to_csv('sub.csv', index=False)

我们可以检查文件的前几行，以确保它看起来是合理的：


In [ ]:
!head sub.csv

当您在 Kaggle 中点击"save version"并等待笔记本运行后，您会看到 `sub.csv` 出现在"Data"选项卡中。点击该文件将显示一个 *Submit* 按钮，允许您提交到竞赛。


## 使用矩阵乘积


我们可以让代码整洁很多……

来看看我们用来获取预测值的最内层计算：


In [ ]:
(val_indep*coeffs).sum(axis=1)

将元素相乘然后跨行相加，与执行矩阵-向量乘积完全相同！Python 使用 `@` 运算符来表示矩阵乘积，PyTorch 张量也支持该运算符。因此，我们可以像这样更简单地复现上述计算：


In [ ]:
val_indep@coeffs

事实证明，这样做速度也快得多，因为 PyTorch 中的矩阵乘法经过了高度优化。

我们来用它替换 `calc_preds` 的实现方式：


In [ ]:
def calc_preds(coeffs, indeps): return torch.sigmoid(indeps@coeffs)

为了进行矩阵-矩阵乘法（我们将在下一节中用到），我们需要将 `coeffs` 转换为列向量（即只有一列的矩阵），我们可以通过向 `torch.rand()` 传递第二个参数 `1` 来实现这一点，表示我们希望系数只有一列：


In [ ]:
def init_coeffs(): return (torch.rand(n_coeff, 1)*0.1).requires_grad_()

我们还需要将因变量转换为列向量，可以通过用特殊值 `None` 对列维度进行索引来实现，这会告诉 PyTorch 在此位置添加一个新维度：


In [ ]:
trn_dep = trn_dep[:,None]
val_dep = val_dep[:,None]

我们现在可以像之前一样训练我们的模型，并确认我们得到了相同的输出……：


In [ ]:
coeffs = train_model(lr=100)

……以及相同的准确度：


In [ ]:
acc(coeffs)

## 一个神经网络


现在我们已经拥有实现神经网络所需的一切。

首先，我们需要为每一层创建系数。第一组系数将接收 `n_coeff` 个输入，并生成 `n_hidden` 个输出。`n_hidden` 的值可以自由选择——数值越大，网络的表达能力越强，但训练速度也会变慢，训练难度也会增加。因此，我们需要一个大小为 `n_coeff` × `n_hidden` 的矩阵。我们会将这些系数除以 `n_hidden`，这样在下一层求和时，结果的数量级与输入保持一致。

第二层则需要接收 `n_hidden` 个输入并输出一个单一结果，因此需要一个 `n_hidden` × `1` 的矩阵。此外，第二层还需要加上一个常数项。


In [ ]:
def init_coeffs(n_hidden=20):
    layer1 = (torch.rand(n_coeff, n_hidden)-0.5)/n_hidden
    layer2 = torch.rand(n_hidden, 1)-0.3
    const = torch.rand(1)[0]
    return layer1.requires_grad_(),layer2.requires_grad_(),const.requires_grad_()

现在我们有了系数，可以创建神经网络了。关键步骤是两个矩阵乘积，`indeps@l1` 和 `res@l2`（其中 `res` 是第一层的输出）。第一层的输出传递给 `F.relu`（这是我们的非线性函数），第二层的输出像之前一样传递给 `torch.sigmoid`。


In [ ]:
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    l1,l2,const = coeffs
    res = F.relu(indeps@l1)
    res = res@l2 + const
    return torch.sigmoid(res)

最后，既然我们现在有了不止一组系数，我们需要添加一个循环来更新每一组：


In [ ]:
def update_coeffs(coeffs, lr):
    for layer in coeffs:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

就这样——我们现在已经准备好训练我们的模型了！


In [ ]:
coeffs = train_model(lr=1.4)

In [ ]:
coeffs = train_model(lr=20)

看起来不错——我们的损失比之前更低了。让我们看看这是否能在验证集上转化为更好的结果：


In [ ]:
acc(coeffs)

在这种情况下，我们的神经网络并没有表现出比线性模型更好的结果。这并不令人惊讶；这个数据集非常小且非常简单，并不是我们期望神经网络表现出色的那种场景。此外，我们的验证集太小，无法可靠地看到太多准确率差异。但关键是，我们现在确切地知道一个真实的神经网络是什么样子的！


## 深度学习


上一节中的神经网络只使用了一个隐藏层，因此并不算是"深度"学习。但我们可以用完全相同的方法让神经网络变得更深，只需添加更多的矩阵乘法即可。

首先，我们需要为每一层创建额外的系数：


In [ ]:
def init_coeffs():
    hiddens = [10, 10]  # <-- set this to the size of each hidden layer you want
    sizes = [n_coeff] + hiddens + [1]
    n = len(sizes)
    layers = [(torch.rand(sizes[i], sizes[i+1])-0.3)/sizes[i+1]*4 for i in range(n-1)]
    consts = [(torch.rand(1)[0]-0.5)*0.1 for i in range(n-1)]
    for l in layers+consts: l.requires_grad_()
    return layers,consts

你会注意到，这里有很多杂乱的常量，用来把随机数控制在合适的范围内。当你稍后训练模型时，你会发现，这些初始值哪怕有最微小的改动，都可能导致模型完全无法训练！这正是深度学习在早期难以取得突破的重要原因之一——要为系数找到一个好的起始点，实在是太挑剔了。如今，我们已经有了应对这一问题的方法，我们将在其他笔记本中学习。

我们深度学习版本的 `calc_preds` 看起来和之前大同小异，但现在我们通过循环遍历每一层，而不是逐一列出它们：


In [ ]:
import torch.nn.functional as F

def calc_preds(coeffs, indeps):
    layers,consts = coeffs
    n = len(layers)
    res = indeps
    for i,l in enumerate(layers):
        res = res@l + consts[i]
        if i!=n-1: res = F.relu(res)
    return torch.sigmoid(res)

我们还需要对 `update_coeffs` 进行小幅更新，因为我们现在已经将 `layers` 和 `consts` 分开了：


In [ ]:
def update_coeffs(coeffs, lr):
    layers,consts = coeffs
    for layer in layers+consts:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

让我们训练我们的模型...


In [ ]:
coeffs = train_model(lr=4)

...并检查其准确性：


In [ ]:
acc(coeffs)

## 最终想法


我们能够从零开始构建一个真正的深度学习模型，并在一个 notebook 中就将其训练到超过 80% 的准确率，这其实相当酷！

在研究和工业界实际使用的"正式"深度学习模型与此非常相似——事实上，如果你翻看任何深度学习模型的源码，都会发现基本步骤是一致的。

实际应用中的模型与我们上面所构建的主要区别在于：

- 初始化和归一化的方式，以确保模型每次都能正确训练
- 正则化（用于避免过拟合）
- 针对具体问题领域的知识对神经网络本身进行改造
- 对更小的批次数据执行梯度下降，而不是对整个数据集操作

这些内容我之后都会陆续更新对应的 notebook，届时会在此处添加相关链接。

如果你觉得这个 notebook 对你有帮助，请记得点击顶部的小向上箭头为它点赞——我很乐意知道自己的工作对大家有所裨益，这也能帮助更多人发现它。（顺便提醒一下，点赞时请确认你浏览的是我的<a href="https://www.kaggle.com/code/jhoward/linear-model-and-neural-net-from-scratch">原始 notebook</a>，而不是你自己复制的版本，否则你的点赞将不会被计入！）如果你有任何问题或想法，欢迎在下方留言——我会阅读每一条评论！


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原始文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误读，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
